In [ ]:
!git clone https://github.com/Tonioboubss/nba-game-analysis-generator.git


Cloning into 'nba-game-analysis-generator'...
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 13 (delta 0), reused 13 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (13/13), 13.50 KiB | 13.50 MiB/s, done.


In [1]:
!pip install -r requirements.txt
!pip uninstall -y torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 16.6 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
import torch
torch.cuda.is_available()

True

In [ ]:
!git clone https://github.com/harvardnlp/boxscore-data.git /tmp/boxscore-data
!mkdir -p data/raw
!tar -jxvf /tmp/boxscore-data/rotowire.tar.bz2 -C data/raw/
!ls data/raw/rotowire

Cloning into '/tmp/boxscore-data'...
remote: Enumerating objects: 42, done.
remote: Total 42 (delta 0), reused 0 (delta 0), pack-reused 42 (from 1)
Receiving objects: 100% (42/42), 28.72 MiB | 20.64 MiB/s, done.
Resolving deltas: 100% (10/10), done.
rotowire/
rotowire/test.json
rotowire/valid.json
rotowire/train.json
test.json  train.json  valid.json


In [ ]:
import json
games = json.load(open("data/raw/rotowire/train.json"))
print(games[0].keys())
print(games[0]["home_line"])
print(list(games[0]["box_score"].keys()))
print(games[0].get("summary", games[0].get("summary_en"))[:15])

dict_keys(['home_name', 'box_score', 'home_city', 'vis_name', 'summary', 'vis_line', 'vis_city', 'day', 'home_line'])
{'TEAM-PTS_QTR2': '20', 'TEAM-FT_PCT': '89', 'TEAM-PTS_QTR1': '28', 'TEAM-PTS_QTR4': '32', 'TEAM-PTS_QTR3': '34', 'TEAM-CITY': 'New York', 'TEAM-PTS': '114', 'TEAM-AST': '11', 'TEAM-LOSSES': '14', 'TEAM-NAME': 'Knicks', 'TEAM-WINS': '16', 'TEAM-REB': '49', 'TEAM-TOV': '17', 'TEAM-FG3_PCT': '35', 'TEAM-FG_PCT': '47'}
['FIRST_NAME', 'MIN', 'FGM', 'REB', 'FG3A', 'PLAYER_NAME', 'AST', 'FG3M', 'OREB', 'TO', 'START_POSITION', 'PF', 'PTS', 'FGA', 'STL', 'FTA', 'BLK', 'DREB', 'FTM', 'FT_PCT', 'FG_PCT', 'FG3_PCT', 'SECOND_NAME', 'TEAM_CITY']
['The', 'Celtics', 'saw', 'great', 'team', 'play', 'in', 'their', 'Christmas', 'Day', 'win', ',', 'and', 'it', 'translated']


In [ ]:
!python src/preprocess.py --raw-dir data/raw/rotowire --out-dir data/processed
!head -c 600 data/processed/train.jsonl

[ok] train: 3398 games -> data/processed/train.jsonl
[ok] valid: 727 games -> data/processed/valid.jsonl
[ok] test: 728 games -> data/processed/test.jsonl
{"id": "12_25_16-Knicks-Celtics", "source": "<record> New York Knicks | PTS_QTR2 | 20 | HOME </record> <record> New York Knicks | FT_PCT | 89 | HOME </record> <record> New York Knicks | PTS_QTR1 | 28 | HOME </record> <record> New York Knicks | PTS_QTR4 | 32 | HOME </record> <record> New York Knicks | PTS_QTR3 | 34 | HOME </record> <record> New York Knicks | PTS | 114 | HOME </record> <record> New York Knicks | AST | 11 | HOME </record> <record> New York Knicks | LOSSES | 14 | HOME </record> <record> New York Knicks | WINS | 16 | HOME </record> <record> New York Knicks | REB | 49 | HOME </recor

In [ ]:
!python src/finetune.py --model t5-small \
  --train data/processed/train.jsonl --valid data/processed/valid.jsonl \
  --output models/bakeoff-t5-small --max-train-examples 500 --epochs 3

!python src/finetune.py --model t5-base \
  --train data/processed/train.jsonl --valid data/processed/valid.jsonl \
  --output models/bakeoff-t5-base --max-train-examples 500 --epochs 3

!python src/finetune.py --model facebook/bart-base \
  --train data/processed/train.jsonl --valid data/processed/valid.jsonl \
  --output models/bakeoff-bart-base --max-train-examples 500 --epochs 3

Loading weights: 100% 131/131 [00:00<00:00, 5015.37it/s]
trainable params: 589,824 || all params: 61,096,448 || trainable%: 0.9654
Map: 100% 500/500 [00:08<00:00, 55.69 examples/s]
Map: 100% 727/727 [00:12<00:00, 59.94 examples/s]
{'loss': '7.541', 'grad_norm': '1.181', 'learning_rate': '0.0002698', 'epoch': '0.3175'}
{'loss': '4.815', 'grad_norm': '0.5772', 'learning_rate': '0.0002381', 'epoch': '0.6349'}
{'loss': '4.35', 'grad_norm': '0.4082', 'learning_rate': '0.0002063', 'epoch': '0.9524'}
 33% 63/189 [00:14<00:23,  5.37it/s]
  0% 0/91 [00:00<?, ?it/s]
  3% 3/91 [00:00<00:05, 17.29it/s]
  5% 5/91 [00:00<00:06, 13.37it/s]
  8% 7/91 [00:00<00:06, 12.06it/s]
 10% 9/91 [00:00<00:06, 11.80it/s]
 12% 11/91 [00:00<00:06, 11.50it/s]
 14% 13/91 [00:01<00:06, 11.34it/s]
 16% 15/91 [00:01<00:06, 11.26it/s]
 19% 17/91 [00:01<00:06, 11.18it/s]
 21% 19/91 [00:01<00:06, 11.05it/s]
 23% 21/91 [00:01<00:06, 11.04it/s]
 25% 23/91 [00:01<00:06, 11.02it/s]
 27% 25/91 [00:02<00:06, 11.00it/s]
 30% 27/9

In [ ]:
!pip uninstall -y torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/rotowire-recap-generator
!cp -r /content/nba-game-analysis-generator/src /content/drive/MyDrive/rotowire-recap-generator/
!cp -r /content/nba-game-analysis-generator/models /content/drive/MyDrive/rotowire-recap-generator/
!cp -r /content/nba-game-analysis-generator/data /content/drive/MyDrive/rotowire-recap-generator/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!python src/predict.py --base-model t5-small --model-dir models/t5-small-bakeoff \
    --input data/processed/test.jsonl --output /tmp/preds-t5-small.jsonl

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Loading weights: 100% 131/131 [00:00<00:00, 5641.04it/s]
[8/728] generated
[16/728] generated
[24/728] generated
[32/728] generated
[40/728] generated
[48/728] generated
[56/728] generated
[64/728] generated
[72/728] generated
[80/728] generated
[88/728] generated
[96/728] generated
[104/728] generated
[112/728] generated
[120/728] generated
[128/728] generated
[136/728] generated
[144/728] generated
[152/728] generated
[160/728] generated
[168/728] generated
[176/728] generated
[184/728] generated
[192/728] generated
[200/728] generated
[208/728] generated
[216/728] generated
[224/728] generated
[232/728] generated
[240/728] generated
[248/728] generated
[256/728] generated
[264/728] generated
[272/728] generated
[280/728] generated
[288/728] generated
[296/728] generated
[304/728] generated
[312/728] generated
[320/728] generated
[328/728] generated
[336/728] generated
[

In [ ]:
import json

with open("/tmp/preds-t5-small.jsonl", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]

for row in rows[:5]:
    print("REAL:", row["target"][:300])
    print("GEN :", row["generated"][:300])
    print("---")

REAL: The Atlanta Hawks ( 46 - 12 ) beat the Orlando Magic ( 19 - 41 ) 95 - 88 on Friday . Al Horford had a good all - around game , putting up 17 points , 13 rebounds , four assists and two steals in a tough matchup against Nikola Vucevic . Kyle Korver was the lone Atlanta starter not to reach double fig
GEN : record Atlanta Hawks - Orlando Magic - 88 - 28 - 29. The Hawks had a 2-0 lead in the first half of the season. Despite the loss, the Hawks were able to win the game.
---
REAL: The Milwaukee Bucks ( 18 - 17 ) defeated the New York Knicks ( 5 - 31 ) 95 - 82 on Sunday at Madison Square Garden in New York . The Bucks were able to have a great night defensively , giving themselves the scoring advantage in all four quarters . The Bucks showed superior shooting , going 46 percen
GEN : New York Knicks have a record of 78 points in the first half of the season. The Milwaukee Bucks are averaging 79 points compared to 78 rebounds in the second half.
---
REAL: The Grizzlies ( 50 ) used a st

In [ ]:
!python src/predict.py --base-model t5-base --model-dir models/t5-base-bakeoff \
    --input data/processed/test.jsonl --output /tmp/preds-t5-base.jsonl

config.json: 100% 1.21k/1.21k [00:00<00:00, 3.23MB/s]

model.safetensors: downloading bytes:  26% 236M/892M [00:02<00:05, 114MB/s, 20.3MB/s  ]
model.safetensors: downloading bytes:  31% 273M/892M [00:03<00:06, 99.9MB/s, 23.2MB/s  ]
model.safetensors: downloading bytes:  34% 303M/892M [00:03<00:06, 86.0MB/s, 24.9MB/s  ]
model.safetensors: downloading bytes:  42% 370M/892M [00:04<00:04, 125MB/s, 29.2MB/s  ]
model.safetensors: downloading bytes:  43% 388M/892M [00:04<00:04, 111MB/s, 30.8MB/s  ]
model.safetensors: reconstructing file:  86% 768M/892M [00:04<00:00, 216MB/s, 51.2MB/s  ]
model.safetensors: downloading bytes: 100% 398M/398M [00:05<00:00, 78.7MB/s, 31.8MB/s  ]
model.safetensors: reconstructing file: 100% 892M/892M [00:05<00:00, 177MB/s, 72.7MB/s  ]
Loading weights: 100% 257/257 [00:00<00:00, 10762.89it/s]
generation_config.json: 100% 147/147 [00:00<00:00, 714kB/s]
[8/728] generated
[16/728] generated
[24/728] generated
[32/728] generated
[40/728] generated
[48/728] generated
[56

In [ ]:
import json

with open("/tmp/preds-t5-base.jsonl", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]

for row in rows[:5]:
    print("REAL:", row["target"][:300])
    print("GEN :", row["generated"][:300])
    print("---")

REAL: The Atlanta Hawks ( 46 - 12 ) beat the Orlando Magic ( 19 - 41 ) 95 - 88 on Friday . Al Horford had a good all - around game , putting up 17 points , 13 rebounds , four assists and two steals in a tough matchup against Nikola Vucevic . Kyle Korver was the lone Atlanta starter not to reach double fig
GEN : The Atlanta Hawks defeated the Orlando Magic 88 - 95 - 88 on Saturday . The Hawks , who were led by Isaiah Thomas , were able to pull off a 98 - 99 win over the Magic , which was the first time they had beaten the Hawks in a playoff game . It was the second time a team dominated by a single - pointe
---
REAL: The Milwaukee Bucks ( 18 - 17 ) defeated the New York Knicks ( 5 - 31 ) 95 - 82 on Sunday at Madison Square Garden in New York . The Bucks were able to have a great night defensively , giving themselves the scoring advantage in all four quarters . The Bucks showed superior shooting , going 46 percen
GEN : The New York Knicks defeated the Milwaukee Bucks 82 - 82 on Saturday 

In [ ]:
!python src/predict.py --base-model facebook/bart-base --model-dir models/bakeoff-bart-base \
    --input data/processed/test.jsonl --output /tmp/preds-bart-base.jsonl

config.json: 100% 1.72k/1.72k [00:00<00:00, 5.84MB/s]

model.safetensors: downloading bytes:   7% 41.6M/558M [00:01<00:17, 29.6MB/s,  894kB/s  ]
model.safetensors: downloading bytes:  12% 68.4M/558M [00:01<00:09, 52.9MB/s, 3.99MB/s  ]
model.safetensors: downloading bytes:  33% 182M/558M [00:02<00:03, 118MB/s, 15.9MB/s  ]
model.safetensors: reconstructing file:  41% 230M/558M [00:02<00:02, 114MB/s, 8.36MB/s  ]  
model.safetensors: downloading bytes:  43% 240M/558M [00:02<00:02, 149MB/s, 19.3MB/s  ]
model.safetensors: downloading bytes:  67% 372M/558M [00:03<00:01, 173MB/s, 31.3MB/s  ]
model.safetensors: reconstructing file:  67% 375M/558M [00:05<00:03, 57.7MB/s, 31.1MB/s  ]
model.safetensors: downloading bytes: 100% 372M/372M [00:05<00:00, 67.8MB/s, 32.0MB/s  ]
model.safetensors: reconstructing file: 100% 558M/558M [00:05<00:00, 102MB/s, 43.8MB/s  ]
Loading weights: 100% 259/259 [00:00<00:00, 18271.07it/s]
[8/728] generated
[16/728] generated
[24/728] generated
[32/728] generated
[40/72

In [ ]:
import json

with open("/tmp/preds-bart-base.jsonl", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]

for row in rows[:5]:
    print("REAL:", row["target"][:300])
    print("GEN :", row["generated"][:300])
    print("---")

REAL: The Atlanta Hawks ( 46 - 12 ) beat the Orlando Magic ( 19 - 41 ) 95 - 88 on Friday . Al Horford had a good all - around game , putting up 17 points , 13 rebounds , four assists and two steals in a tough matchup against Nikola Vucevic . Kyle Korver was the lone Atlanta starter not to reach double fig
GEN : The Atlanta Hawks defeated the Orlando Magic 95 - 88 on Monday night . The Hawks ( 2 - 0 ) were led by Tobias Harris , who had a game - high 22 points , while the Magic ( 1 - 1 ) were held to 10 points , 11 rebounds , and three assists . The Magic ( 0 - 4 ) were able to hold off the Magic , who have
---
REAL: The Milwaukee Bucks ( 18 - 17 ) defeated the New York Knicks ( 5 - 31 ) 95 - 82 on Sunday at Madison Square Garden in New York . The Bucks were able to have a great night defensively , giving themselves the scoring advantage in all four quarters . The Bucks showed superior shooting , going 46 percen
GEN : The Milwaukee Bucks defeated the New York Knicks ( 2 - 0 ) at Madison

In [ ]:
import json

def load(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

t5_small = load("/tmp/preds-t5-small.jsonl")
t5_base = load("/tmp/preds-t5-base.jsonl")
bart_base = load("/tmp/preds-bart-base.jsonl")

for i in range(10):
    print(f"=== Match {i+1} ===")
    print("REAL     :", t5_small[i]["target"][:250])
    print("T5-SMALL :", t5_small[i]["generated"][:250])
    print("T5-BASE  :", t5_base[i]["generated"][:250])
    print("BART-BASE:", bart_base[i]["generated"][:250])
    print()

=== Match 1 ===
REAL     : The Atlanta Hawks ( 46 - 12 ) beat the Orlando Magic ( 19 - 41 ) 95 - 88 on Friday . Al Horford had a good all - around game , putting up 17 points , 13 rebounds , four assists and two steals in a tough matchup against Nikola Vucevic . Kyle Korver wa
T5-SMALL : record Atlanta Hawks - Orlando Magic - 88 - 28 - 29. The Hawks had a 2-0 lead in the first half of the season. Despite the loss, the Hawks were able to win the game.
T5-BASE  : The Atlanta Hawks defeated the Orlando Magic 88 - 95 - 88 on Saturday . The Hawks , who were led by Isaiah Thomas , were able to pull off a 98 - 99 win over the Magic , which was the first time they had beaten the Hawks in a playoff game . It was the
BART-BASE: The Atlanta Hawks defeated the Orlando Magic 95 - 88 on Monday night . The Hawks ( 2 - 0 ) were led by Tobias Harris , who had a game - high 22 points , while the Magic ( 1 - 1 ) were held to 10 points , 11 rebounds , and three assists . The Magic ( 

=== Match 2 ===
REAL

In [ ]:
from transformers import AutoTokenizer
import json

tok = AutoTokenizer.from_pretrained("facebook/bart-base")
lengths = []
with open("data/processed/train.jsonl", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        lengths.append(len(tok(row["source"])["input_ids"]))

import statistics
print("médiane:", statistics.median(lengths))
print("moyenne:", statistics.mean(lengths))
print("max:", max(lengths))
print("% > 512:", sum(l > 512 for l in lengths) / len(lengths) * 100)
print("% > 768:", sum(l > 768 for l in lengths) / len(lengths) * 100)

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

médiane: 7126.0
moyenne: 7238.735432607416
max: 9025
% > 512: 100.0
% > 768: 100.0


In [ ]:
import json

def count_unk(path):
    total, unk = 0, 0
    with open(path.replace("processed", "raw/rotowire").replace(".jsonl", ".json"), encoding="utf-8") as f:
        games = json.load(f)
    for game in games:
        box = game["box_score"]
        home_city, vis_city = game.get("home_city", ""), game.get("vis_city", "")
        for idx, city in box.get("TEAM_CITY", {}).items():
            total += 1
            if city not in (home_city, vis_city):
                unk += 1
    print(f"{path}: {unk}/{total} joueurs en UNK ({unk/total*100:.1f}%)")

count_unk("data/processed/train.jsonl")

data/processed/train.jsonl: 0/87024 joueurs en UNK (0.0%)


In [ ]:
!python src/preprocess.py --raw-dir data/raw/rotowire --out-dir data/processed --top-n-players 8

[ok] train: 3398 games -> data/processed/train.jsonl
[ok] valid: 727 games -> data/processed/valid.jsonl
[ok] test: 728 games -> data/processed/test.jsonl


In [ ]:
!python src/preprocess.py --raw-dir data/raw/rotowire --out-dir data/processed --top-n-players 3

from transformers import AutoTokenizer
import json, statistics

tok = AutoTokenizer.from_pretrained("facebook/bart-base")
lengths = []
with open("data/processed/train.jsonl", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        lengths.append(len(tok(row["source"])["input_ids"]))

print("médiane:", statistics.median(lengths))
print("moyenne:", statistics.mean(lengths))
print("max:", max(lengths))
print("% > 512:", sum(l > 512 for l in lengths) / len(lengths) * 100)
print("% > 768:", sum(l > 768 for l in lengths) / len(lengths) * 100)
print("% > 1024:", sum(l > 1024 for l in lengths) / len(lengths) * 100)

[ok] train: 3398 games -> data/processed/train.jsonl
[ok] valid: 727 games -> data/processed/valid.jsonl
[ok] test: 728 games -> data/processed/test.jsonl
médiane: 987.0
moyenne: 988.5876986462625
max: 1064
% > 512: 100.0
% > 768: 99.94114184814596
% > 1024: 4.79693937610359


In [ ]:
from transformers import AutoTokenizer
import json, statistics

tok = AutoTokenizer.from_pretrained("facebook/bart-base")
lengths = []
with open("data/processed/train.jsonl", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        lengths.append(len(tok(row["target"])["input_ids"]))

print("médiane:", statistics.median(lengths))
print("moyenne:", statistics.mean(lengths))
print("max:", max(lengths))
print("% > 256:", sum(l > 256 for l in lengths) / len(lengths) * 100)

médiane: 348.0
moyenne: 354.04561506768687
max: 814
% > 256: 72.36609770453207


In [ ]:
print("% > 384:", sum(l > 384 for l in lengths) / len(lengths) * 100)
print("% > 512:", sum(l > 512 for l in lengths) / len(lengths) * 100)

% > 384: 39.199529134785166
% > 512: 11.653914067098293


In [ ]:
#Refinetuning with right truncature on both input and output
!python src/finetune.py --model t5-small \
  --train data/processed/train.jsonl --valid data/processed/valid.jsonl \
  --output models/bakeoff-t5-small-v2 --max-train-examples 500 --epochs 3

!python src/finetune.py --model t5-base \
  --train data/processed/train.jsonl --valid data/processed/valid.jsonl \
  --output models/bakeoff-t5-base-v2 --max-train-examples 500 --epochs 3

!python src/finetune.py --model facebook/bart-base \
  --train data/processed/train.jsonl --valid data/processed/valid.jsonl \
  --output models/bakeoff-bart-base-v2 --max-train-examples 500 --epochs 3

Loading weights: 100% 257/257 [00:00<00:00, 14186.54it/s]
trainable params: 1,769,472 || all params: 224,673,024 || trainable%: 0.7876
Map: 100% 500/500 [00:01<00:00, 262.09 examples/s]
Map: 100% 727/727 [00:02<00:00, 316.53 examples/s]
  0% 0/189 [00:00<?, ?it/s]Traceback (most recent call last):
  File "/content/src/finetune.py", line 180, in <module>
    main()
    ~~~~^^
  File "/content/src/finetune.py", line 170, in main
    trainer.train()
    ~~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/transformers/trainer.py", line 1458, in train
    return inner_training_loop(
        args=args,
    ...<2 lines>...
        ignore_keys_for_eval=ignore_keys_for_eval,
    )
  File "/usr/local/lib/python3.13/dist-packages/transformers/trainer.py", line 1540, in _inner_training_loop
    self._run_epoch(
    ~~~~~~~~~~~~~~~^
        model=model,
        ^^^^^^^^^^^^
    ...<9 lines>...
        steps_trained_in_current_epoch=steps_trained_in_current_epoch,
        ^^^^^^^^^^^^^^^

In [ ]:
!python src/predict.py --base-model t5-small --model-dir models/bakeoff-t5-small-v2 \
    --input data/processed/test.jsonl --output /tmp/preds-t5-small-v2.jsonl
!python src/predict.py --base-model t5-base --model-dir models/bakeoff-t5-base-v2 \
    --input data/processed/test.jsonl --output /tmp/preds-t5-base-v2.jsonl
!python src/predict.py --base-model facebook/bart-base --model-dir models/bakeoff-bart-base-v2 \
    --input data/processed/test.jsonl --output /tmp/preds-bart-base-v2.jsonl

Loading weights: 100% 131/131 [00:00<00:00, 4104.44it/s]
[8/728] generated
[16/728] generated
[24/728] generated
[32/728] generated
[40/728] generated
[48/728] generated
[56/728] generated
[64/728] generated
[72/728] generated
[80/728] generated
[88/728] generated
[96/728] generated
[104/728] generated
[112/728] generated
[120/728] generated
[128/728] generated
[136/728] generated
[144/728] generated
[152/728] generated
[160/728] generated
[168/728] generated
[176/728] generated
[184/728] generated
[192/728] generated
[200/728] generated
[208/728] generated
[216/728] generated
[224/728] generated
[232/728] generated
[240/728] generated
[248/728] generated
[256/728] generated
[264/728] generated
[272/728] generated
[280/728] generated
[288/728] generated
[296/728] generated
[304/728] generated
[312/728] generated
[320/728] generated
[328/728] generated
[336/728] generated
[344/728] generated
[352/728] generated
[360/728] generated
[368/728] generated
[376/728] generated
[384/728] genera

In [ ]:
!python src/evaluate_generation.py --predictions /tmp/preds-t5-small-v2.jsonl
!python src/evaluate_generation.py --predictions /tmp/preds-t5-base-v2.jsonl
!python src/evaluate_generation.py --predictions /tmp/preds-bart-base-v2.jsonl

BLEU: 0.04
Approx. content coverage (NOT the paper's CS metric — see module docstring): 0.238 over 728 examples
That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.
BLEU: 0.42
Approx. content coverage (NOT the paper's CS metric — see module docstring): 0.134 over 728 examples
BLEU: 5.78
Approx. content coverage (NOT the paper's CS metric — see module docstring): 0.283 over 728 examples


In [ ]:
!python src/finetune.py --model facebook/bart-base --train data/processed/train.jsonl \
    --valid data/processed/valid.jsonl --output models/bart-base-final \
    --epochs 8 --early-stopping-patience 2 --max-source-len 1024 --max-target-len 512


config.json: 100% 1.72k/1.72k [00:00<00:00, 4.01MB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 90.9MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 106MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 120MB/s]

model.safetensors: downloading bytes:   6% 32.9M/558M [00:01<00:17, 30.6MB/s,  610kB/s  ]
model.safetensors: downloading bytes:  10% 54.4M/558M [00:01<00:09, 53.2MB/s, 3.18MB/s  ]
model.safetensors: downloading bytes:  19% 103M/558M [00:01<00:04, 113MB/s, 7.03MB/s  ]  
model.safetensors: downloading bytes:  28% 157M/558M [00:02<00:02, 146MB/s, 12.4MB/s  ]
model.safetensors: downloading bytes:  40% 225M/558M [00:02<00:01, 185MB/s, 17.7MB/s  ]
model.safetensors: reconstructing file:  52% 290M/558M [00:02<00:01, 173MB/s, 21.2MB/s  ]
model.safetensors: downloading bytes:  55% 308M/558M [00:02<00:01, 206MB/s, 25.3MB/s  ]
model.safetensors: downloading bytes:  64% 354M/558M [00:02<00:00, 215MB/s, 28.7MB/s  ]
model.safetensors: downloading bytes: 100% 372M/372M [00:03<00:00, 116MB/s, 

ValueError: mount failed

In [ ]:
!zip -r bart-base-final.zip models/bart-base-final
from google.colab import files
files.download('bart-base-final.zip')

  adding: models/bart-base-final/ (stored 0%)
  adding: models/bart-base-final/checkpoint-425/ (stored 0%)
  adding: models/bart-base-final/checkpoint-425/rng_state.pth (deflated 26%)
  adding: models/bart-base-final/checkpoint-425/scaler.pt (deflated 64%)
  adding: models/bart-base-final/checkpoint-425/trainer_state.json (deflated 74%)
  adding: models/bart-base-final/checkpoint-425/optimizer.pt (deflated 8%)
  adding: models/bart-base-final/checkpoint-425/README.md (deflated 66%)
  adding: models/bart-base-final/checkpoint-425/tokenizer_config.json (deflated 50%)
  adding: models/bart-base-final/checkpoint-425/training_args.bin (deflated 54%)
  adding: models/bart-base-final/checkpoint-425/tokenizer.json (deflated 82%)
  adding: models/bart-base-final/checkpoint-425/scheduler.pt (deflated 61%)
  adding: models/bart-base-final/checkpoint-425/adapter_model.safetensors (deflated 7%)
  adding: models/bart-base-final/checkpoint-425/adapter_config.json (deflated 60%)
  adding: models/bart-

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!nvidia-smi

Sat Sep 19 20:09:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!python src/finetune.py --model t5-base \
  --train data/processed/train.jsonl --valid data/processed/valid.jsonl \
  --output models/bakeoff-t5-base-v2 --max-train-examples 500 --epochs 3 \
  --batch-size 2 --gradient-accumulation-steps 4

Loading weights: 100% 257/257 [00:00<00:00, 3856.31it/s]
trainable params: 1,769,472 || all params: 224,673,024 || trainable%: 0.7876
Map: 100% 500/500 [00:01<00:00, 305.28 examples/s]
Map: 100% 727/727 [00:02<00:00, 312.05 examples/s]
{'loss': '46.91', 'grad_norm': '16.32', 'learning_rate': '0.0002698', 'epoch': '0.32'}
{'loss': '18.63', 'grad_norm': '2.346', 'learning_rate': '0.0002381', 'epoch': '0.64'}
{'loss': '14.9', 'grad_norm': '1.658', 'learning_rate': '0.0002063', 'epoch': '0.96'}
 33% 63/189 [01:57<05:12,  2.48s/it]
  0% 0/364 [00:00<?, ?it/s]
  1% 2/364 [00:00<00:37,  9.58it/s]
  1% 3/364 [00:00<00:52,  6.93it/s]
  1% 4/364 [00:00<01:00,  5.96it/s]
  1% 5/364 [00:00<01:03,  5.68it/s]
  2% 6/364 [00:01<01:07,  5.34it/s]
  2% 7/364 [00:01<01:09,  5.12it/s]
  2% 8/364 [00:01<01:11,  4.96it/s]
  2% 9/364 [00:01<01:12,  4.90it/s]
  3% 10/364 [00:01<01:14,  4.73it/s]
  3% 11/364 [00:02<01:13,  4.78it/s]
  3% 12/364 [00:02<01:12,  4.88it/s]
  4% 13/364 [00:02<01:12,  4.81it/s]
  4

In [ ]:
!zip -r bakeoff-t5-base-v2.zip models/bakeoff-t5-base-v2
from google.colab import files
files.download('bakeoff-t5-base-v2.zip')

  adding: models/bakeoff-t5-base-v2/ (stored 0%)
  adding: models/bakeoff-t5-base-v2/checkpoint-63/ (stored 0%)
  adding: models/bakeoff-t5-base-v2/checkpoint-63/rng_state.pth (deflated 26%)
  adding: models/bakeoff-t5-base-v2/checkpoint-63/scaler.pt (deflated 64%)
  adding: models/bakeoff-t5-base-v2/checkpoint-63/trainer_state.json (deflated 60%)
  adding: models/bakeoff-t5-base-v2/checkpoint-63/optimizer.pt (deflated 8%)
  adding: models/bakeoff-t5-base-v2/checkpoint-63/README.md (deflated 66%)
  adding: models/bakeoff-t5-base-v2/checkpoint-63/tokenizer_config.json (deflated 83%)
  adding: models/bakeoff-t5-base-v2/checkpoint-63/training_args.bin (deflated 54%)
  adding: models/bakeoff-t5-base-v2/checkpoint-63/tokenizer.json (deflated 75%)
  adding: models/bakeoff-t5-base-v2/checkpoint-63/scheduler.pt (deflated 62%)
  adding: models/bakeoff-t5-base-v2/checkpoint-63/adapter_model.safetensors (deflated 7%)
  adding: models/bakeoff-t5-base-v2/checkpoint-63/adapter_config.json (deflated 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!zip -r bakeoff-bart-base-v2.zip models/bakeoff-bart-base-v2
from google.colab import files
files.download('bakeoff-bart-base-v2.zip')

  adding: models/bakeoff-bart-base-v2/ (stored 0%)
  adding: models/bakeoff-bart-base-v2/checkpoint-63/ (stored 0%)
  adding: models/bakeoff-bart-base-v2/checkpoint-63/rng_state.pth (deflated 26%)
  adding: models/bakeoff-bart-base-v2/checkpoint-63/scaler.pt (deflated 64%)
  adding: models/bakeoff-bart-base-v2/checkpoint-63/trainer_state.json (deflated 61%)
  adding: models/bakeoff-bart-base-v2/checkpoint-63/optimizer.pt (deflated 8%)
  adding: models/bakeoff-bart-base-v2/checkpoint-63/README.md (deflated 66%)
  adding: models/bakeoff-bart-base-v2/checkpoint-63/tokenizer_config.json (deflated 50%)
  adding: models/bakeoff-bart-base-v2/checkpoint-63/training_args.bin (deflated 54%)
  adding: models/bakeoff-bart-base-v2/checkpoint-63/tokenizer.json (deflated 82%)
  adding: models/bakeoff-bart-base-v2/checkpoint-63/scheduler.pt (deflated 62%)
  adding: models/bakeoff-bart-base-v2/checkpoint-63/adapter_model.safetensors (deflated 7%)
  adding: models/bakeoff-bart-base-v2/checkpoint-63/adap

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!zip -r bakeoff-t5-small-v2.zip models/bakeoff-t5-small-v2
from google.colab import files
files.download('bakeoff-t5-small-v2.zip')

  adding: models/bakeoff-t5-small-v2/ (stored 0%)
  adding: models/bakeoff-t5-small-v2/checkpoint-63/ (stored 0%)
  adding: models/bakeoff-t5-small-v2/checkpoint-63/rng_state.pth (deflated 26%)
  adding: models/bakeoff-t5-small-v2/checkpoint-63/scaler.pt (deflated 64%)
  adding: models/bakeoff-t5-small-v2/checkpoint-63/trainer_state.json (deflated 61%)
  adding: models/bakeoff-t5-small-v2/checkpoint-63/optimizer.pt (deflated 7%)
  adding: models/bakeoff-t5-small-v2/checkpoint-63/README.md (deflated 66%)
  adding: models/bakeoff-t5-small-v2/checkpoint-63/tokenizer_config.json (deflated 82%)
  adding: models/bakeoff-t5-small-v2/checkpoint-63/training_args.bin (deflated 54%)
  adding: models/bakeoff-t5-small-v2/checkpoint-63/tokenizer.json (deflated 75%)
  adding: models/bakeoff-t5-small-v2/checkpoint-63/scheduler.pt (deflated 62%)
  adding: models/bakeoff-t5-small-v2/checkpoint-63/adapter_model.safetensors (deflated 7%)
  adding: models/bakeoff-t5-small-v2/checkpoint-63/adapter_config.js

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
!python src/predict.py \
    --model-dir models/bart-base-final \
    --input data/processed/test.jsonl \
    --output data/processed/test_predictions.jsonl

config.json: 100% 1.72k/1.72k [00:00<00:00, 5.46MB/s]

model.safetensors: downloading bytes:  16% 86.7M/558M [00:00<00:03, 149MB/s, 6.00MB/s  ]
model.safetensors: downloading bytes:  23% 128M/558M [00:01<00:02, 169MB/s, 10.2MB/s  ] 
model.safetensors: reconstructing file:  16% 88.5M/558M [00:01<00:06, 71.6MB/s, 7.37MB/s  ]
model.safetensors: downloading bytes:  29% 162M/558M [00:01<00:02, 142MB/s, 13.8MB/s  ]
model.safetensors: downloading bytes:  34% 192M/558M [00:01<00:02, 144MB/s, 16.0MB/s  ]
model.safetensors: downloading bytes:  45% 253M/558M [00:02<00:02, 147MB/s, 20.8MB/s  ]
model.safetensors: downloading bytes:  50% 280M/558M [00:02<00:01, 142MB/s, 23.4MB/s  ]
model.safetensors: downloading bytes:  56% 312M/558M [00:02<00:01, 148MB/s, 26.1MB/s  ]
model.safetensors: downloading bytes:  64% 358M/558M [00:02<00:01, 152MB/s, 29.3MB/s  ]
model.safetensors: downloading bytes: 100% 372M/372M [00:03<00:00, 121MB/s, 31.9MB/s  ]
model.safetensors: reconstructing file: 100% 558M/558M [00:

In [3]:
!python src/evaluate_generation.py \
    --predictions data/processed/test_predictions.jsonl \
    --out data/processed/test_scores.jsonl

That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.
BLEU: 9.56
Approx. content coverage (NOT the paper's CS metric — see module docstring): 0.365 over 728 examples
Per-example scores written to data/processed/test_scores.jsonl


In [4]:
from huggingface_hub import login
login()  # colle un token avec accès "Write" — https://huggingface.co/settings/tokens

In [5]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel

MODEL_DIR = "models/bart-base-final"
REPO_ID = "antoineboubouu/bart-base-rotowire-recap"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
base_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-base")
model = PeftModel.from_pretrained(base_model, MODEL_DIR)

model.push_to_hub(REPO_ID)
tokenizer.push_to_hub(REPO_ID)

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  31%|###1      | 1.11MB / 3.55MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/antoineboubouu/bart-base-rotowire-recap/commit/8c1b9a88e4c0764c8701a867a8e7de46c19a92f1', commit_message='Upload tokenizer', commit_description='', oid='8c1b9a88e4c0764c8701a867a8e7de46c19a92f1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/antoineboubouu/bart-base-rotowire-recap', endpoint='https://huggingface.co', repo_type='model', repo_id='antoineboubouu/bart-base-rotowire-recap'), pr_revision=None, pr_num=None)

In [7]:
from huggingface_hub import upload_file

README = """---
license: mit
base_model: facebook/bart-base
tags:
- lora
- peft
- data-to-text
- nba
datasets:
- rotowire
metrics:
- bleu
---

# BART-base fine-tuné sur RotoWire (box-score NBA -> compte-rendu)

Adaptateur LoRA (r=16, alpha=32, cible q_proj/v_proj) fine-tuné sur `facebook/bart-base`
pour générer un compte-rendu textuel de match NBA à partir d'une box-score structurée
(dataset RotoWire, Wiseman et al. 2017 — harvardnlp/boxscore-data).

## Entrée attendue
Box-score linéarisée en `<record> entité | stat | valeur | HOME|VIS </record>` —
stats d'équipe complètes + les 3 joueurs les plus utilisés par équipe (par minutes)
sur 6 stats (MIN/PTS/REB/AST/STL/BLK). Voir `preprocess.py` du repo GitHub pour le
détail du format.

## Résultats (test set RotoWire, 728 matchs)
- BLEU : 9.56
- Couverture de contenu (proxy approximatif, pas la métrique CS du papier) : 0.365

Comparaison à la littérature (Wiseman et al. 2017, Table 2) : baseline à gabarits
6.78 BLEU, modèles neuronaux à mécanisme de copie 12.96-14.49 BLEU. Ce modèle se
situe au-dessus du baseline naïf, en dessous des modèles à copie explicite — attendu,
puisque ce modèle n'a pas de mécanisme de copie dédié et reçoit une entrée compressée
(3 joueurs/équipe, 6 stats) plutôt que la box-score complète.

## Limites connues
- Entraîné et évalué uniquement sur des matchs jusqu'au 29/03/2017 (date d'arrêt du dataset).
- Pas de mécanisme de copie explicite pour les valeurs numériques.
- Sélection de contenu par heuristique fixe (top-3 joueurs par minutes), pas apprise.

Code complet : [lien vers ton repo GitHub]
"""

with open("README.md", "w") as f:
    f.write(README)

upload_file(path_or_fileobj="README.md", path_in_repo="README.md", repo_id=REPO_ID)

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/antoineboubouu/bart-base-rotowire-recap/commit/1b5c1a8e68af914f5be9defe4f01faf9eb419fb2', commit_message='Upload README.md with huggingface_hub', commit_description='', oid='1b5c1a8e68af914f5be9defe4f01faf9eb419fb2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/antoineboubouu/bart-base-rotowire-recap', endpoint='https://huggingface.co', repo_type='model', repo_id='antoineboubouu/bart-base-rotowire-recap'), pr_revision=None, pr_num=None)

In [8]:
import json, random

with open("data/processed/test.jsonl", encoding="utf-8") as f:
    games = [json.loads(l) for l in f]

random.Random(0).shuffle(games)
with open("data/test_sample.jsonl", "w", encoding="utf-8") as f:
    for g in games[:50]:
        f.write(json.dumps(g, ensure_ascii=False) + "\n")

In [10]:
from huggingface_hub import create_repo, upload_file

SPACE_ID = "antoineboubouu/rotowire-recap-demo"
create_repo(SPACE_ID, repo_type="space", space_sdk="gradio", space_hardware="zero-a10g")

upload_file(path_or_fileobj="src/app.py", path_in_repo="app.py", repo_id=SPACE_ID, repo_type="space")
upload_file(path_or_fileobj="requirements.txt", path_in_repo="requirements.txt", repo_id=SPACE_ID, repo_type="space")
upload_file(path_or_fileobj="data/test_sample.jsonl", path_in_repo="data/test_sample.jsonl", repo_id=SPACE_ID, repo_type="space")

CommitInfo(commit_url='https://huggingface.co/spaces/antoineboubouu/rotowire-recap-demo/commit/c65d84de5acea551e9912f42ac1f3b4a4d0da273', commit_message='Upload data/test_sample.jsonl with huggingface_hub', commit_description='', oid='c65d84de5acea551e9912f42ac1f3b4a4d0da273', pr_url=None, repo_url=RepoUrl('https://huggingface.co/spaces/antoineboubouu/rotowire-recap-demo', endpoint='https://huggingface.co', repo_type='space', repo_id='antoineboubouu/rotowire-recap-demo'), pr_revision=None, pr_num=None)

In [11]:
from huggingface_hub import create_repo, upload_file

SPACE_ID = "antoineboubouu/rotowire-recap-demo"
create_repo(SPACE_ID, repo_type="space", space_sdk="gradio")

upload_file(path_or_fileobj="src/app.py", path_in_repo="app.py", repo_id=SPACE_ID, repo_type="space")
upload_file(path_or_fileobj="requirements.txt", path_in_repo="requirements.txt", repo_id=SPACE_ID, repo_type="space")
upload_file(path_or_fileobj="data/test_sample.jsonl", path_in_repo="data/test_sample.jsonl", repo_id=SPACE_ID, repo_type="space")

HfHubHTTPError: Client error '402 Payment Required' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-6aafa2d3-5303624e11b5c2c366684f07;4a9b7e1a-8012-4dfa-9a3e-c280e5f3cc95)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

Static Spaces are free for everyone, but hosting Gradio and Docker Spaces on free cpu-basic requires a PRO subscription. Subscribe at https://huggingface.co/pro

In [12]:
from huggingface_hub import upload_file

upload_file(path_or_fileobj="src/app.py", path_in_repo="app.py", repo_id="antoineboubouu/rotowire-recap-demo", repo_type="space")

CommitInfo(commit_url='https://huggingface.co/spaces/antoineboubouu/rotowire-recap-demo/commit/f0aabe2ca2d8ac33e886e6556786dbbec48c224e', commit_message='Upload app.py with huggingface_hub', commit_description='', oid='f0aabe2ca2d8ac33e886e6556786dbbec48c224e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/spaces/antoineboubouu/rotowire-recap-demo', endpoint='https://huggingface.co', repo_type='space', repo_id='antoineboubouu/rotowire-recap-demo'), pr_revision=None, pr_num=None)